# **HOMEWORK 3**

In this homework you will explore the practical challenges of deploying and adapting large-scale Transformer models under real-world hardware constraints.


In **Task 1** , you will load a 3-billion-parameter language model and directly observe the memory pressure it places on a GPU. You will then progressively apply three **Optimization techniques** (4-bit quantization, static KV caching with `torch.compile`, precision improvement) and measure their impact on memory footprint, inference speed, and accuracy. Finally, you will perform structured and unstructured pruning on a smaller model directly on its weights, and compare the pruned model against the original in terms of size, speed, and quality.


In **Task 2** , you will fine-tune a pretrained BERT model on **Natural Language Inference (NLI)**: lassifying whether a hypothesis is an entailment , contradiction , or neutral given a premise, without updating all 110 million parameters. Instead, you will implement two parameter-efficient fine-tuning (PEFT) methods: **Bottleneck Adapters** , which insert small MLP modules inside each Transformer block, and **LoRA** , which injects trainable low-rank matrices alongside the attention weights.


By the end, you will compare both PEFT methods side-by-side on accuracy, training time, and the fraction of parameters actually trained, and see concretely what each efficient technique trades off.



# Setup

Run the following cells to import and configure everything you need for this homework.

In [ ]:
!pip install transformers
!pip install -U datasets huggingface_hub fsspec
!pip install accelerate -U
!pip install bitsandbytes
!pip install peft evaluate
!pip install torchao --upgrade -q

  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel
import numpy as np
import matplotlib.pyplot as plt
import math
import seaborn as sns
from datasets import load_dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from transformers import BitsAndBytesConfig
import time
import subprocess
import evaluate
import warnings
import os
warnings.filterwarnings("ignore")
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BertConfig,
    BertTokenizer,
    BertModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
)

In [ ]:
# Define device
use_gpu = True

device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [ ]:
import psutil
# Utility function for showing gpu-usage information

def show_device_status():
    if str(device) == "cuda":
      result = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
      print(result.stdout)
    else:
      print(f"CPU usage: {psutil.cpu_percent()}%")

      mem = psutil.virtual_memory()
      print(f"RAM used: {mem.percent}%")
      print(f"RAM available: {mem.available / 1024**3:.2f} GB")
      print(f"CPU cores: {os.cpu_count()}")


In [ ]:
# Specify the seed, don't change
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch

    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [ ]:
import gc

def clean_memory():
  # delete model variables if they exist
  for var in ["model", "model_quant4", "model_quant_statickv", "past_key_values", "outputs", "inputs"]:
      if var in globals():
          del globals()[var]

  # garbage collect
  gc.collect()

  # clear CUDA cache
  if str(device) == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

  print("GPU memory cleared")

# Task 1: Inference with Memory-Efficient Transformers (10 pts)

In this section, you will interact with a large-scale language model (3B parameters) and encounter limitations of running such models on consumer-grade hardware. You will progressively apply model with multiple quanitzation  techniques then observe their effect on GPU memory usage and model performance. You can open the resources tab of collab to see the consumption graphically as well. Refer to these resources:

- [HF Optimization Techniques](https://huggingface.co/docs/transformers/en/llm_optims)
- [BitsAndBytes Quantization](https://huggingface.co/docs/transformers/main_classes/quantization)

Note: This HW is gpu intensive task running on cpu should also posible but can be slow. Some tips are below.

*   Use the clean_memory() function if the system shows out of memory error.
*   CPU runtime will take longer in quantization and at inference time may crash as well.

Restart session if the error pursues or contact any TA for support.

In [ ]:
model_name = "tiiuae/Falcon3-3B-Base"

## 1.1 Load LLM and observe the memory (0.5 pt)

Try loading a large model (3B parameters) and observe resources. This will take some time as it will download a 6GB model.


### 1.1.1 Load the model (0.5 pts)


In [ ]:
try:
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
except Exception as e:
    print("Error occurred:", str(e))

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/91.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/362k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

Print the memory footprint and device status now

you might have to change the "model" to the vairable name used by you in the above cell

In [ ]:
print(f"{model.get_memory_footprint() / 1024**3:.2f} GB")
show_device_status()

6.01 GB
Sat Jun 20 15:26:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             28W /   70W |    6265MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------------

Run to clean the memory used

In [ ]:
clean_memory()
show_device_status()

GPU memory cleared
Sat Jun 20 15:26:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             28W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

## 1.2 Optimisation 1 (1.5 pts)



### 1.2.1 Try loading the same model with Quantization (0.5 pts)

In [ ]:
quant_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_quant4 = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map=device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Print the memory footprint and device status now

In [ ]:
print(f"{model_quant4.get_memory_footprint() / 1024**3:.2f} GB")
show_device_status()

2.63 GB
Sat Jun 20 15:26:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             28W /   70W |    3041MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------------

Inferencing with prompt "What's the famous city in Italy?" and measuring of inference time plus device memory using `show_device_status()`.

In [ ]:
prompt = "What's the famous city in Italy?"
inputs = tokenizer(prompt, return_token_type_ids=False, return_tensors="pt").to(device)

print("Before generation:")
show_device_status()

start = time.time()
with torch.no_grad():
    outputs = model_quant4.generate(**inputs, max_new_tokens=20)
end = time.time()

print("\nAfter generation:")
show_device_status()

print(f"\nGenerated: {tokenizer.decode(outputs[0])}")
print(f"Inference time: {end - start:.2f} seconds")

Before generation:


[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Sat Jun 20 15:26:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             29W /   70W |    3041MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



After generation:
Sat Jun 20 15:26:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             46W /   70W |    3253MiB /  15360MiB |     32%      Default |
|                                         |                        |                  N/A |
+----------------------------

### 1.2.2 Multiple Choice Question (1 pts)

Do you see the difference in memory footprint and memory allocation GPU VRAM what could be the possible reasons? (Choose all that apply, separate multiple answers with space or comma)

 A. PyTorch’s CUDA Caching Allocator retains the VRAM freed from deleted temporary 16-bit tensors as "Reserved Memory".

 B. The model must permanently store both the original 16-bit weights and the new 4-bit weights in VRAM simultaneously to perform inference.

 C. CUDA context overhead inherently consumes between 0.5 GB to 1.0 GB of VRAM, regardless of the model's actual size.

 D. The quantization process requires the CPU RAM to perfectly match the GPU VRAM capacity to prevent bottlenecking.

In [ ]:
# Write your answer below
print("A")

A


## 1.3 - Optimisation 2 (2 pts)




### 1.3.1 Try loading the model with static KV caching also use torch.compile to avoid "graph break". Combine it with `Optimisation 1`. (1 pts) [link text](https://huggingface.co/docs/transformers/kv_cache#:~:text=compile%20to%20accelerate%20generation.,without%20needing%20to%20modify%20it.)

In [ ]:
model_quant_statickv = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map=device)
model_quant_statickv.generation_config.cache_implementation = "static"
model_quant_statickv.forward = torch.compile(model_quant_statickv.forward, mode="reduce-overhead", fullgraph=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferencing with prompt "What's the famous city in Italy?" and measuring of inference time plus device memory using `show_device_status()`.

The first run might take a lot longer but subsequent runs on the same and different query will be faster. [Link](https://huggingface.co/docs/transformers/perf_torch_compile)

In [ ]:
prompt = "What's the famous city in Italy?"
inputs = tokenizer(prompt, return_token_type_ids=False, return_tensors="pt").to(device)

print("Before generation:")
show_device_status()

start = time.time()
outputs = model_quant_statickv.generate(**inputs, max_new_tokens=20)
end = time.time()

print("\nAfter generation:")
show_device_status()

print("\nGenerated:", tokenizer.decode(outputs[0]))
print(f"Inference time: {end - start:.2f} seconds")

[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Before generation:
Sat Jun 20 15:26:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             28W /   70W |    5933MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

W0620 15:27:40.366000 6121 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode



After generation:
Sat Jun 20 15:29:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             43W /   70W |    6225MiB /  15360MiB |     36%      Default |
|                                         |                        |                  N/A |
+----------------------------

### 1.3.2 Multiple Choice Question? (1 pts)

Based on the observed behavior of quantization, torch.compile and KV cache strategies, which of the following statements are true? (Choose all that apply, separate multiple answers with space or comma)

 A. The first inference takes significantly longer due to initial setup and graph compilation, while subsequent queries execute faster because they utilize the cached calculations.

 B. A major advantage of using a Static KV cache is that it avoids Out of Memory (OOM) issues by maintaining a fixed memory allocation.

 C. Dynamic KV caching inherently prevents Out of Memory (OOM) issues by strictly limiting the number of tokens that can be mapped during inference.

 D. Subsequent inference runs become progressively slower because the continuously growing cache creates processing bottlenecks.

In [ ]:
# Write your answer below
print("A,B")

A,B


Run to clean the memory used

In [ ]:
clean_memory()
show_device_status()

GPU memory cleared
Sat Jun 20 15:29:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             42W /   70W |     475MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

## 1.4 - Optimisation vs Performance (3 pts)

Evaluate the tradeoff between quantization and accuracy on a classification task. Use [SST-2](https://huggingface.co/datasets/stanfordnlp/sst2) dataset to generate labels. Report the accuracy score for the performance evaluation.

#### 1.4.1 Write an evaluate_model function to calcuate the accuracy of the model on the dataset then use the function to evaluate the model with Optimisation 1(from Task 1.2.1) and optimisation 2(from Task 1.3.1) on the dataset and compare the performance (1 pts)

In [ ]:
dataset = load_dataset("stanfordnlp/sst2", split="validation[:50]")

# You should not modify the function arguments
def evaluate_model(model, tokenizer, dataset):
    correct = 0
    for item in dataset:
        inputs = tokenizer(item["sentence"], return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
        pred = tokenizer.decode(outputs[0][-1]).strip()
        if pred == str(item["label"]):
            correct += 1
    return correct / len(dataset)

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Load the models from the tasks above task 1.2.1 and task 1.3.1 in the below cell and print the accuracy

Note - Copy only the first cell with model load part and the qunatization applied. You don't need to copy memory output and text generation here.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Copy the cell 1.2.1 here
quant_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_quant4 = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map=device)

acc_quant = evaluate_model(model_quant4, tokenizer, dataset)
print(f"Quantized model accuracy: {acc_quant:.2f}")

clean_memory()

# Copy the cell 1.3.1 here
model_quant_statickv = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map=device)
model_quant_statickv.generation_config.cache_implementation = "static"
model_quant_statickv.forward = torch.compile(model_quant_statickv.forward, mode="reduce-overhead", fullgraph=False)

acc_quant_flash = evaluate_model(model_quant_statickv, tokenizer, dataset)
print(f"Quantized + Static KV cache accuracy: {acc_quant_flash:.2f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Quantized model accuracy: 0.00
GPU memory cleared


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

W0620 15:30:57.162000 6121 torch/_dynamo/convert_frame.py:1743] [15/8] torch._dynamo hit config.recompile_limit (8)
W0620 15:30:57.162000 6121 torch/_dynamo/convert_frame.py:1743] [15/8]    function: 'matmul_4bit' (/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:369)
W0620 15:30:57.162000 6121 torch/_dynamo/convert_frame.py:1743] [15/8]    last reason: 15/7: tensor 'A' size mismatch at index 1. expected 1, actual 13
W0620 15:30:57.162000 6121 torch/_dynamo/convert_frame.py:1743] [15/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0620 15:30:57.162000 6121 torch/_dynamo/convert_frame.py:1743] [15/8] To diagnose recompilation issues, see https://docs.pytorch.org/docs/main/user_guide/torch_compiler/compile/programming_model.recompilation.html


Quantized + Static KV cache accuracy: 0.00


Run to clean the memory used

In [ ]:
clean_memory()
show_device_status()

GPU memory cleared
Sat Jun 20 15:31:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             37W /   70W |     475MiB /  15360MiB |     44%      Default |
|                                         |                        |                  N/A |
+----------------------------

### 1.4.2 Try to redefine the optimisation config for the model to improve performance (better precision) while maintaining a similar CPU/GPU usage and print the accuracy (1 pts)

In [ ]:
# TODO: Your code here
quant_config_improved = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model_improved = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config_improved, device_map=device)

acc_improved = evaluate_model(model_improved, tokenizer, dataset)
print(f"Improved quantized model accuracy: {acc_improved}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Improved quantized model accuracy: 0.0


### 1.4.3 Multiple Choice Question? (1 pts)

Based on the performance comparison of the three quantized models on the 200-sample SST subset, which of the following statements are correct? (Choose all that apply, separate multiple answers with space or comma)

A. All three models reached nearly identical accuracy levels (89–90%), with Model 3 exhibiting a marginal gain.

B. Model 2 improved numerical stability by computing in bfloat16, while Model 3 reduced runtime overhead with a static KV cache.

C. Model 3 achieved a massive accuracy improvement over the other models by utilizing a static KV cache and torch.compile.

D. Model 2 minimized runtime generation overhead through a static KV cache and torch.compile.

E. The models varied drastically in their accuracy, primarily because Model 2 lacked torch.compile during generation.

In [ ]:
# Write your answer below
print("A,B")

A,B


## 1.5 - Model pruning (3 pts)

Model pruning can be done in different ways.

*   Structured
*   Unstructured
*   Magnitude

There are two way of pruning first load the whole model in GPU and apply the above pruning methods, second load the safe tensors prune them and save a copy. We will implement the second way and use "Qwen/Qwen2.5-0.5B-Instruct" model. We chose this model so that you can compare size & accuracy of the original and pruned model easily.

Note: We are doing zero-shot pruning that means we will directly run inference after doing the model pruning and not retrain/fine-tune. If the accuracy drops don't panic. The evaluation will be done based on correct implementation of pruning.

### 1.5.1 - Use snapshot_download from hugging face hub to download the model (0.25 pts)

Printing the path that the model gets saved in and also listing down the files inside the directory. There should be files like 'config.json', 'model.safetensors' etc. The model size is close to 1 GB so it may take few seconds or a minute to dowload depends on your internet speed.

In [ ]:
# TODO: Your code here
# use qwen_model_dir = ...
from huggingface_hub import snapshot_download
qwen_model_dir = snapshot_download(repo_id="Qwen/Qwen2.5-0.5B-Instruct")

print(qwen_model_dir)
print(os.listdir(qwen_model_dir))

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775
['merges.txt', 'model.safetensors', '.gitattributes', 'generation_config.json', 'config.json', 'tokenizer_config.json', 'vocab.json', 'README.md', 'tokenizer.json', 'LICENSE']


Run the below block to check if the safe tensor file is loading fine or not and define the output directory

In [ ]:
from safetensors import safe_open

with safe_open(qwen_model_dir + "/model.safetensors", framework="pt") as f:
    keys = list(f.keys())

print("Total tensors:", len(keys))
print("Sample keys:", keys[:10])

out_dir = "/content/qwen_pruned_model"
os.makedirs(out_dir, exist_ok=True)

Total tensors: 290
Sample keys: ['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_proj.bias']


### 1.5.2 Fill the below functions to prune models weight and skip layers. (1 pts)

Take out threshold of the weights by taking its absolute value mean multiplied with a ratio passed. (use w.abs())

Soft pruning example - check if any item is above the threshold we change the magnitude by 1/2 of the current value.

Magnitude pruning example - check if any item is below the threshold we change it to 0

Layer skip - If the name of the layer has any of the skip layer values it should return True otherwise False. (Check the name to understand better). Skip layer is an array with layers you want to skip "[10, 11]"

In [ ]:
def soft_prune(w, ratio=0.5):
    threshold = w.abs().mean() * ratio
    mask = w.abs() > threshold
    w[mask] = w[mask] * 0.5
    return w

def magnitude_prune(w, ratio=0.5):
    threshold = w.abs().mean() * ratio
    mask = w.abs() < threshold
    w[mask] = 0.0
    return w

def layer_skip(name, skip_layers):
    return any(f".layers.{layer_idx}." in name for layer_idx in skip_layers)

### 1.5.3 Use the safe_open to load the mode.safetensors just like the couple of blocks before and call the above functions on each layer weights (1 pts)

First call layer_skip if the layer is not skipped perform soft pruning if name of the layer has "mlp" or "dense" in it otherwise go for magnitude pruning

In [ ]:
from safetensors import safe_open

pruned = {}
skip_layers = [10, 11]  # example: drop last layers for speed test or try with any other layers of your choice

# Re-open the file and keep the block indented
with safe_open(qwen_model_dir + "/model.safetensors", framework="pt") as f:
    keys = list(f.keys())
    for name in keys:
        tensor = f.get_tensor(name)
        if layer_skip(name, skip_layers):
            continue
        if "mlp" in name or "dense" in name:
            pruned[name] = soft_prune(tensor)
        else:
            pruned[name] = magnitude_prune(tensor)

Save the pruned safe tensors

In [ ]:
from safetensors.torch import save_file

save_file(pruned, out_dir + "/model.safetensors")

print("Saved to:", out_dir)

Saved to: /content/qwen_pruned_model


Copy important files to load the pruned model successfully later

In [ ]:
import shutil

files_to_copy = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "generation_config.json",
    "special_tokens_map.json"
]

for f in files_to_copy:
    src = os.path.join(qwen_model_dir, f)

    if os.path.exists(src):
        shutil.copy(src, out_dir)

Compare the size of the folders

In [ ]:
def folder_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 ** 2)

original_size = folder_size_mb(qwen_model_dir)
print("Original model size (MB):", original_size)

pruned_size = folder_size_mb(out_dir)
print("Pruned model size (MB):", pruned_size)

Original model size (MB): 953.296781539917
Pruned model size (MB): 892.1484251022339


Check inference time of original and puned model and also print generated output.


In [ ]:
qwen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype="auto",
    device_map=device
)

pruned_qwen_model = AutoModelForCausalLM.from_pretrained(
    out_dir,
    device_map=device
)

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

inputs = qwen_tokenizer("What's the famous city in Italy? Please answer in English", return_tensors="pt").to(device)

print("Before generation from qwen model:")
show_device_status()

start = time.time()
out = qwen_model.generate(**inputs, max_new_tokens=20)
end = time.time()

print("\nAfter generation from qwen model:")
show_device_status()

print(f"Generation: {qwen_tokenizer.decode(out[0])}")
print(f"Inference time for qwen model: {end - start:.2f} seconds")

print("Before generation from pruned qwen model:")
show_device_status()

start = time.time()
pruned_model_out = pruned_qwen_model.generate(**inputs, max_new_tokens=20)
end = time.time()

print("\nAfter generation from pruned qwen model:")
show_device_status()

print(f"Generation: {qwen_tokenizer.decode(pruned_model_out[0])}")
print(f"Inference time for qwen model: {end - start:.2f} seconds")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] Qwen2ForCausalLM LOAD REPORT from: /content/qwen_pruned_model
Key                                                   | Status  | 
------------------------------------------------------+---------+-
model.layers.{10, 11}.self_attn.q_proj.bias           | MISSING | 
model.layers.{10, 11}.post_attention_layernorm.weight | MISSING | 
model.layers.{10, 11}.self_attn.k_proj.bias           | MISSING | 
model.layers.{10, 11}.self_attn.q_proj.weight         | MISSING | 
model.layers.{10, 11}.mlp.up_proj.weight              | MISSING | 
model.layers.{10, 11}.mlp.gate_proj.weight            | MISSING | 
model.layers.{10, 11}.self_attn.k_proj.weight         | MISSING | 
model.layers.{10, 11}.self_attn.o_proj.weight         | MISSING | 
model.layers.{10, 11}.input_layernorm.weight          | MISSING | 
model.layers.{10, 11}.self_attn.v_proj.bias           | MISSING | 
model.layers.{10, 11}.mlp.down_proj.weight            | MISSING | 
model.layers.{10, 11}.self_attn.v_proj.weight       

Before generation from qwen model:
Sat Jun 20 15:32:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             34W /   70W |    5237MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

### 1.5.5 Use the same evaluation function from above with the same data to check the qwen and pruned qwen models accuracy print them (0.25 pts)

In [ ]:
acc_qwen = evaluate_model(qwen_model, qwen_tokenizer, dataset)
print(f"Qwen Instruct 0.5B model accuracy: {acc_qwen:.2f}")

acc_pruned_qwen = evaluate_model(pruned_qwen_model, qwen_tokenizer, dataset)
print(f"Prudned Qwen Instruct 0.5B model accuracy: {acc_pruned_qwen:.2f}")

Qwen Instruct 0.5B model accuracy: 0.02
Prudned Qwen Instruct 0.5B model accuracy: 0.00


### 1.5.5 Multiple Choice Question (0.5 pts)

How does pruning affect a model? (Choose all that apply, separate multiple answers with space or comma)

A. Smaller

B. Faster

C. Bloated

D. Efficient

In [ ]:
# Write your answer below
print("A,B,D")

A,B,D


#2: Parameter-Efficient Fine-Tuning with BERT (10 Points)

In this homework you will fine-tune a pretrained **BERT-base** model on **Natural Language Inference (NLI)**: the task of deciding, given a *premise* and a *hypothesis*, whether the hypothesis is an *entailment*, *contradiction*, or *neutral* with respect to the premise.

Rather than updating all 110 million parameters of BERT (which is expensive and often unnecessary), you will explore two **parameter-efficient fine-tuning (PEFT)** methods that achieve competitive accuracy by training only a tiny fraction of the model's weights:

1. **Bottleneck Adapters**: small MLP modules inserted *inside* each transformer layer
2. **LoRA (Low-Rank Adaptation)**: low-rank matrix pairs added *alongside* existing attention weight matrices

By the end, you will evaluate both models on a held-out test set and compare their accuracy, training time, and parameter efficiency.

## Setup and label mapping

Run the cell below to import everything needed for this task.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

id2label = {0: "entailment", 1: "neutral", 2: "contradiction"}
label2id = {"entailment": 0, "neutral": 1, "contradiction": 2}
NUM_LABELS = 3
MODEL_CHECKPOINT = "bert-base-uncased"

Using device: cuda
PyTorch version: 2.11.0+cu128


## Dataset: SNLI (Stanford Natural Language Inference)

The SNLI dataset contains pairs of sentences: a **premise** and a **hypothesis**. The task is to classify their relationship as one of:

- `entailment`: the hypothesis logically follows from the premise
- `neutral`: the hypothesis is neither confirmed nor contradicted
- `contradiction`: the hypothesis directly contradicts the premise

We load 10,000 training examples, 2,000 validation examples, and 2,000 test examples. Examples with label `-1` (unannotatable) are filtered out.


In [ ]:
raw_dataset = load_dataset("stanfordnlp/snli")
raw_dataset = raw_dataset.filter(lambda x: x["label"] != -1)

TRAIN_SIZE = 10_000
EVAL_SIZE  = 2_000
TEST_SIZE  = 2_000

train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
eval_dataset  = raw_dataset["validation"].shuffle(seed=42).select(range(EVAL_SIZE))
test_dataset  = raw_dataset["test"].shuffle(seed=42).select(range(TEST_SIZE))

print("Dataset splits:")
print(f"  Train : {len(train_dataset):,} examples")
print(f"  Val   : {len(eval_dataset):,} examples")
print(f"  Test  : {len(test_dataset):,} examples")

print("\nSample example:")
ex = train_dataset[0]
print(f"  Premise    : {ex['premise']}")
print(f"  Hypothesis : {ex['hypothesis']}")
print(f"  Label      : {id2label[ex['label']]} ({ex['label']})")

print("\nLabel distribution in training set:")
from collections import Counter
counts = Counter(train_dataset["label"])
for lbl_id, count in sorted(counts.items()):
    print(f"  {id2label[lbl_id]:<15}: {count:,} ({count/len(train_dataset)*100:.1f}%)")

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/550152 [00:00<?, ? examples/s]

Dataset splits:
  Train : 10,000 examples
  Val   : 2,000 examples
  Test  : 2,000 examples

Sample example:
  Premise    : A group of people riding a yellow roller coaster.
  Hypothesis : A group of people are riding a roller coaster.
  Label      : entailment (0)

Label distribution in training set:
  entailment     : 3,352 (33.5%)
  neutral        : 3,291 (32.9%)
  contradiction  : 3,357 (33.6%)


##2.1: Tokenization (2 Points)

Before fine-tuning, we need to convert raw text into token IDs that BERT can process.

For NLI, BERT takes **two sentences** as a single input in the format:
[CLS] premise [SEP] hypothesis [SEP]


The `AutoTokenizer` from HuggingFace handles this automatically when you pass both strings.

### What you need to implement

Complete `tokenize_function(examples)` so that it tokenizes the `"premise"` and `"hypothesis"` fields together using the pre-loaded `tokenizer`.

**Requirements:**
- Pass both `premise` and `hypothesis` to the tokenizer
- Enable `truncation=True` so sequences longer than the model's max length are cut
- Set `max_length=128`

**Expected output** after running the sanity check:


Keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids shape: torch.Size([128])   # (may vary with padding)

**Useful docs:** [`AutoTokenizer.__call__`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizerBase.__call__)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    # TODO: Tokenize premise and hypothesis together as a sentence pair.
    # Use truncation=True and max_length=128.
    # The tokenizer accepts batched input when called with lists.
    # raise NotImplementedError("Implement tokenize_function")
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        max_length=128,
        truncation=True
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval  = eval_dataset.map(tokenize_function,  batched=True)
tokenized_test  = test_dataset.map(tokenize_function,  batched=True)

cols_to_remove = ["premise", "hypothesis"]
tokenized_train = tokenized_train.remove_columns(cols_to_remove)
tokenized_eval  = tokenized_eval.remove_columns(cols_to_remove)
tokenized_test  = tokenized_test.remove_columns(cols_to_remove)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval  = tokenized_eval.rename_column("label",  "labels")
tokenized_test  = tokenized_test.rename_column("label",  "labels")

tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")
tokenized_test.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenization complete.")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

## Part 1: Fine-Tuning with Bottleneck Adapters

### Background

A **Bottleneck Adapter** is a small trainable module inserted after the feed-forward sublayer of each Transformer block. The base BERT weights are **frozen**, only the adapter weights (and the classifier head) are updated during training.

Each adapter has this structure:


input (hidden_size)
→
Linear(hidden_size to bottleneck_dim # down-project)
→
ReLU
→
Linear(bottleneck_dim to hidden_size # up-project)
→
input (residual connection)
→
output (hidden_size)

The residual connection is critical: it ensures that at initialization (when `up_project` weights are zero), the adapter is an **identity function** and does not disturb the pretrained representations.


##2.2: Implement `BotlneckAdapter.forward` (3 Points)

The class skeleton and `__init__` are provided. You need to implement the `forward` method.

**What to fill in:**
1. Save the input as `residual`
2. Pass `x` through `self.down_project`
3. Apply `self.activation`
4. Pass through `self.up_project`
5. Return `x + residual` (the residual connection)

**Sample dimensions** (for `hidden_size=768`, `bottleneck_dim=64`):
- Input `x`: `[batch_size, seq_len, 768]`
- After down-project: `[batch_size, seq_len, 64]`
- After up-project: `[batch_size, seq_len, 768]`
- Output: `[batch_size, seq_len, 768]` (same as input)

**Note:** The `up_project` weights are initialized to zero, so at the start of training `forward(x) == x` exactly. This is intentional.

**Useful docs:** [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html), [`nn.ReLU`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

In [ ]:
class BotlneckAdapter(nn.Module):
    def __init__(self, hidden_size: int, bottleneck_dim: int = 64):
        super().__init__()
        self.down_project = nn.Linear(hidden_size, bottleneck_dim)
        self.activation   = nn.ReLU()
        self.up_project   = nn.Linear(bottleneck_dim, hidden_size)

        nn.init.zeros_(self.up_project.weight)
        nn.init.zeros_(self.up_project.bias)

    def forward(self, x):
        # TODO: implement the bottleneck adapter forward pass
        residual = x
        x = self.down_project(x)
        x = self.activation(x)
        x = self.up_project(x)
        return x + residual


##2.3: Implement `inject_adapters` (1 Point)

`inject_adapters` modifies a BERT model **in-place** by replacing each layer's `output` module with an `OutputWithAdapter` wrapper — a new module that runs the original output and then passes the result through a `BotlneckAdapter`.

**What to fill in inside `inject_adapters`:**

Loop over `model.bert.encoder.layer`. For each `layer`:
1. Save `layer.output` as `original_output`
2. Create a new `BotlneckAdapter(hidden_size, bottleneck_dim)`
3. Replace `layer.output` with `OutputWithAdapter(original_output, adapter)`

The `OutputWithAdapter` class and `freeze_base_model` helper are already provided — read them carefully to understand the expected structure before implementing the loop.

**Sanity check:** After calling `inject_adapters`, `model.bert.encoder.layer[0].output` should be an instance of `OutputWithAdapter`.

In [ ]:
def inject_adapters(model, bottleneck_dim: int = 64):
    hidden_size = model.config.hidden_size

    for layer in model.bert.encoder.layer:
        original_output = layer.output

        class OutputWithAdapter(nn.Module):
            def __init__(self, orig, adapter):
                super().__init__()
                self.orig    = orig
                self.adapter = adapter

            def forward(self, hidden_states, input_tensor):
                x = self.orig(hidden_states, input_tensor)
                x = self.adapter(x)
                return x


        # TODO: create a BotlneckAdapter with the correct hidden_size and bottleneck_dim
        # TODO: replace layer.output with an OutputWithAdapter wrapping original_output and the adapter
        adapter = BotlneckAdapter(hidden_size, bottleneck_dim)
        layer.output = OutputWithAdapter(original_output,adapter)

    return model

In [ ]:
def freeze_base_model(model):
    for param in model.parameters():
        param.requires_grad = False

    for layer in model.bert.encoder.layer:
        for param in layer.output.adapter.parameters():
            param.requires_grad = True
        for param in layer.output.orig.LayerNorm.parameters():
            param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    return model


def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total

print("Adapter classes defined.")

Adapter classes defined.


### Adapter Training (1 Point)

The cell below loads BERT, injects adapters, freezes the base model, and trains using HuggingFace `Trainer`. Just run it and observe the trainable parameter count and training loss. Due to GPU constraints, we recomend that you test your code by initially just training for **1 epoch**, and then increase it to **10** after verification

You should see that **< 5%** of parameters are trainable.

In [ ]:
print("=" * 60)
print("Fine-tuning with Adapters:")
print("=" * 60)

BOTTLENECK_DIM = 64

base_model_adapter = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

adapter_model = inject_adapters(base_model_adapter, bottleneck_dim=BOTTLENECK_DIM)
adapter_model = freeze_base_model(adapter_model)

trainable, total = count_params(adapter_model)
print(f"\nTrainable parameters : {trainable:,}")
print(f"Total parameters     : {total:,}")
print(f"Trainable %          : {100 * trainable / total:.3f}%")

adapter_training_args = TrainingArguments(
    output_dir="./results_adapter",
    num_train_epochs=15,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=1e-3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

adapter_trainer = Trainer(
    model=adapter_model,
    args=adapter_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nStarting Adapter training...")
start_adapter = time.time()
adapter_trainer.train()
adapter_time = time.time() - start_adapter
print(f"\nAdapter training finished in {adapter_time / 60:.1f} minutes")

Fine-tuning with Adapters:


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Trainable parameters : 1,210,371
Total parameters     : 110,674,179
Trainable %          : 1.094%

Starting Adapter training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.109118,1.098052,0.358000
2,1.100989,1.104376,0.358000
3,1.105541,1.103088,0.358000
4,1.099249,1.108178,0.308000
5,1.101001,1.097100,0.358000
6,1.099813,1.099272,0.334000
7,1.099690,1.097162,0.358000
8,1.100615,1.099330,0.334000
9,1.099381,1.098400,0.358000
10,1.099870,1.098052,0.358000



Adapter training finished in 5.9 minutes


##2.4 Multiple Choice Question (1 Point)
Read the `freeze_base_model` function above carefully, then answer the following question.


After calling `freeze_base_model(model)`, which statement best describes what is trainable?

A. The entire BERT model is trainable, `freeze_base_model` only freezes the embedding layer.

B. Only the classification head is trainable; the adapter weights are frozen along with the rest of BERT.

C. The adapter modules, the LayerNorm inside each layer's original output block, and the classification head are trainable, all other BERT weights are frozen.

D. Only the adapter modules are trainable; the LayerNorm and classification head are frozen to preserve pretrained representations.

In [ ]:
# Write your answer below
print("C")

C


## Part 2: Fine-Tuning with LoRA (Low-Rank Adaptation)

### Background

**LoRA** injects trainable **low-rank matrices** into the weight matrices of the attention layers, while keeping all original weights frozen.

For a weight matrix `W ∈ ℝ^{d×k}`, LoRA adds: W' = W + BA

where `B ∈ ℝ^{d×r}` and `A ∈ ℝ^{r×k}`, and `r << min(d, k)` is the **rank** (a small number like 4 or 8).

- `A` is initialized with random Gaussian values
- `B` is initialized to zero (so `BA = 0` at the start — same identity trick as adapters)
- The scaling factor `alpha / r` controls the magnitude of the update

This reduces the number of trainable parameters from `d×k` to `r×(d+k)`.

**In practice:** LoRA is applied only to the `query` and `value` projection matrices in each attention head.

**Useful docs:** [`LoraConfig`](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig), [`get_peft_model`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.get_peft_model)

##2.5: Configure LoRA (2 Points)

Use the `peft` library to configure and wrap BERT with LoRA.

**What to fill in inside `LoraConfig(...)`:**

| Parameter | Description | Recommended Value to use |
|---|---|---|
| `task_type` | Type of task (use `TaskType.SEQ_CLS` for classification) | `TaskType.SEQ_CLS` |
| `r` | LoRA rank — number of low-rank dimensions | `LORA_RANK` (defined above) |
| `lora_alpha` | Scaling factor (conventionally `2 × r`) | `16` |
| `lora_dropout` | Dropout applied to LoRA layers | `0.1` |
| `target_modules` | Which weight matrices to apply LoRA to | `["query", "value"]` |
| `bias` | Whether to train bias terms | `"none"` |

**Expected output** from `lora_model.print_trainable_parameters()`:

trainable params: ~300,000 || all params: ~109,000,000 || trainable%: ~0.28%

(exact numbers may vary slightly)

In [ ]:
print("=" * 60)
print("PART 2: Fine-tuning with LoRA")
print("=" * 60)

LORA_RANK = 8

base_model_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
# TODO: Fill in the LoraConfig arguments.
# See the markdown cell above for what each parameter should be.
lora_config = LoraConfig(
      task_type = TaskType.SEQ_CLS,
      r = LORA_RANK,
      lora_alpha = 16,
      lora_dropout = 0.1,
      target_modules = ["query", "value"],
      bias = "none",
)


lora_model = get_peft_model(base_model_lora, lora_config)
lora_model.print_trainable_parameters()

lora_training_args = TrainingArguments(
    output_dir="./results_lora",
    num_train_epochs=15,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nStarting LoRA training...")
start_lora = time.time()
lora_trainer.train()
lora_time = time.time() - start_lora
print(f"\nLoRA training finished in {lora_time / 60:.1f} minutes")

PART 2: Fine-tuning with LoRA


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!



Starting LoRA training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.766134,0.618340,0.749500
2,0.604914,0.528086,0.802000
3,0.527995,0.500521,0.810500
4,0.478653,0.500626,0.826500
5,0.453210,0.471587,0.824500
6,0.419414,0.472361,0.826500
7,0.404639,0.506156,0.831000
8,0.382205,0.498477,0.830500
9,0.333791,0.556138,0.820000
10,0.329607,0.503687,0.830500



LoRA training finished in 6.7 minutes


## Evaluation & Comparison

Run the cell below to evaluate both models on the test set and print a side-by-side comparison table.

You should observe that both methods reach competitive accuracy despite training only a tiny fraction of the total parameters. Think about:

- Which method trains more parameters?
- Which method trains faster?
- Which method achieves higher accuracy on this task?

In [ ]:
print("=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

print("\nEvaluating Adapter model...")
adapter_results = adapter_trainer.evaluate(tokenized_test)

print("Evaluating LoRA model...")
lora_results = lora_trainer.evaluate(tokenized_test)

adapter_trainable, adapter_total = count_params(adapter_model)
lora_trainable,    lora_total    = count_params(lora_model)

print("\n")
print("=" * 65)
print(f"{'COMPARISON SUMMARY':^65}")
print("=" * 65)
print(f"{'Metric':<38} {'Adapter':>12} {'LoRA':>12}")
print("-" * 65)
print(f"{'Test Accuracy':<38} {adapter_results['eval_accuracy']:>12.4f} {lora_results['eval_accuracy']:>12.4f}")
print(f"{'Test Loss':<38} {adapter_results['eval_loss']:>12.4f} {lora_results['eval_loss']:>12.4f}")
print(f"{'Training Time (min)':<38} {adapter_time/60:>12.1f} {lora_time/60:>12.1f}")
print(f"{'Trainable Params':<38} {adapter_trainable:>12,} {lora_trainable:>12,}")
print(f"{'Total Params':<38} {adapter_total:>12,} {lora_total:>12,}")
print(f"{'Trainable %':<38} {100*adapter_trainable/adapter_total:>11.3f}% {100*lora_trainable/lora_total:>11.3f}%")
print(f"{'Bottleneck dim / LoRA rank':<38} {BOTTLENECK_DIM:>12} {LORA_RANK:>12}")
print("=" * 65)

EVALUATION ON TEST SET

Evaluating Adapter model...


Training Loss,Validation Loss,Epoch,Accuracy
1.098687,1.098641,15,0.333500


Evaluating LoRA model...


Training Loss,Validation Loss,Epoch,Accuracy
0.287643,0.515406,15,0.818500




                       COMPARISON SUMMARY                        
Metric                                      Adapter         LoRA
-----------------------------------------------------------------
Test Accuracy                                0.3335       0.8185
Test Loss                                    1.0986       0.5154
Training Time (min)                             5.9          6.7
Trainable Params                          1,210,371      297,219
Total Params                            110,674,179  109,781,766
Trainable %                                  1.094%       0.271%
Bottleneck dim / LoRA rank                       64            8


## Qualitative Predictions

The cell below runs both trained models on a few hand-crafted examples and prints the predicted label and confidence scores.

No implementation needed, although it is recommended that you include your own examples observe where the two models agree or disagree.

In [ ]:
def predict_nli(premise, hypothesis, model, tokenizer, device):
    model.eval()
    inputs = tokenizer(
        premise,
        hypothesis,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    pred  = np.argmax(probs)

    print(f"  Premise    : {premise}")
    print(f"  Hypothesis : {hypothesis}")
    print(f"  Prediction : {id2label[pred]}  (confidence: {probs[pred]*100:.1f}%)")
    print(f"  All probs  : entailment={probs[0]:.3f}  neutral={probs[1]:.3f}  contradiction={probs[2]:.3f}")

adapter_model.to(device)
lora_model.to(device)

examples = [
    {
        "premise":    "A man is playing guitar on a stage in front of a large crowd.",
        "hypothesis": "A musician is performing live.",
    },
    {
        "premise":    "Two dogs are running through a grassy field.",
        "hypothesis": "The animals are sleeping indoors.",
    },
    {
        "premise":    "A woman is reading a book in a café.",
        "hypothesis": "Someone is inside a building.",
    },
]

for i, ex in enumerate(examples, 1):
    print(f"\nExample {i}")
    print("  --- Bottleneck Adapter ---")
    predict_nli(ex["premise"], ex["hypothesis"], adapter_model, tokenizer, device)
    print("  --- LoRA ---")
    predict_nli(ex["premise"], ex["hypothesis"], lora_model, tokenizer, device)
    print()


Example 1
  --- Bottleneck Adapter ---
  Premise    : A man is playing guitar on a stage in front of a large crowd.
  Hypothesis : A musician is performing live.
  Prediction : entailment  (confidence: 33.7%)
  All probs  : entailment=0.337  neutral=0.331  contradiction=0.332
  --- LoRA ---
  Premise    : A man is playing guitar on a stage in front of a large crowd.
  Hypothesis : A musician is performing live.
  Prediction : neutral  (confidence: 93.0%)
  All probs  : entailment=0.063  neutral=0.930  contradiction=0.007


Example 2
  --- Bottleneck Adapter ---
  Premise    : Two dogs are running through a grassy field.
  Hypothesis : The animals are sleeping indoors.
  Prediction : entailment  (confidence: 33.7%)
  All probs  : entailment=0.337  neutral=0.331  contradiction=0.332
  --- LoRA ---
  Premise    : Two dogs are running through a grassy field.
  Hypothesis : The animals are sleeping indoors.
  Prediction : contradiction  (confidence: 99.8%)
  All probs  : entailment=0.000  